In [2]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [4]:
dataset = load_dataset('surrey-nlp/BESSTIE-CW-26')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/711k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/70.6k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/415k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3747 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/313 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2183 [00:00<?, ? examples/s]

In [5]:
test_df = pd.DataFrame(dataset['test'])
test_df['Sarcasm'] = test_df['Sarcasm'].astype(int)

In [6]:
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model = AutoModelForSequenceClassification.from_pretrained('models/roberta_sarcasm')
model = model.to(device)
model.eval()

print('Model Loaded!')

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model Loaded!


In [7]:
# Slang dictionary for replacement

slang_dict = {
    # Australian slang
    'arvo' : 'afternoon',
    'servo' : 'petrol station',
    'maccas' : 'mcdonalds',
    'brekkie' : 'breakfast',
    'barbie' : 'barbecue',
    'reckon' : 'think',
    'heaps' : 'very',
    'mate' : 'friend',
    'onya' : 'good on you',
    'ute' : 'truck',
    'bonzer' : 'excellent',
    'crikey' : 'wow',
    'yeah nah' : 'no',
    'nah yeah' : 'yes',
    # Indian English
    'coz' : 'because',
    'yaar' : 'friend',
    'bhai' : 'brother',
    'timepass' : 'waste of time',
    'bakwaas' : 'nonsense',
    'jugaad' : 'workaround',
    'prepone' : 'bring forward',
}

In [27]:
import re

def replace_slang(text):
    text_lower = text.lower()
    for slang, replacement in slang_dict.items():
        # Using word boundary to avoid replacing substrings
        # Without this, "ute" would match inside "absolutely" -> "absoltruckly"
        # re.escape handles slang phrases with special characters safely
        pattern = r'\b' + re.escape(slang) + r'\b'
        text_lower = re.sub(pattern, replacement, text_lower)
    return text_lower

In [28]:
def get_confidence(texts):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors='pt'
    )
    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
    return probs

In [32]:
# test
sample = "Yeah nah mate, the arvo session was heaps good onya."
print("Original: ", sample)
print("Replaced: ", replace_slang(sample))

# test edge case
sample2 = "Absolutely lovely routes and ultimately good."
print("\nEdge case original: ", sample2)
print("Edge case replaced: ", replace_slang(sample2))

Original:  Yeah nah mate, the arvo session was heaps good onya.
Replaced:  no friend, the afternoon session was very good good on you.

Edge case original:  Absolutely lovely routes and ultimately good.
Edge case replaced:  absolutely lovely routes and ultimately good.


In [33]:
# Filter the test sentences that has slang words

au_test = test_df[test_df['variety'] == 'en-AU'].copy()
in_test = test_df[test_df['variety'] == 'en-IN'].copy()

In [1]:
# Check if there is any slags which belongs to specific vareity
# Only sentences with slang are useful for adversarial testing
# because we need something to substitute
def has_slang(text):
    text_lower = text.lower()
    for slang in slang_dict.keys():
        if slang in text_lower:
            return True
    return False

In [35]:
au_slang = au_test[au_test['text'].apply(has_slang)]
in_slang = in_test[in_test['text'].apply(has_slang)]

In [36]:
print("en-AU sentences with slang:", len(au_slang))
print("en-IN sentences with slang:", len(in_slang))

en-AU sentences with slang: 82
en-IN sentences with slang: 39


In [37]:
all_slang = pd.concat([au_slang, in_slang], ignore_index=True)
print("Total sentences to test: ", len(all_slang))

Total sentences to test:  121


In [41]:
results = []

for _, row in all_slang.iterrows():
    original_text  = row['text']
    perturbed_text = replace_slang(original_text)

    orig_probs = get_confidence([original_text])[0]
    orig_pred  = int(np.argmax(orig_probs))
    orig_conf  = float(max(orig_probs))

    pert_probs = get_confidence([perturbed_text])[0]
    pert_pred  = int(np.argmax(pert_probs))
    pert_conf  = float(max(pert_probs))

    results.append({
        'variety' : row['variety'],
        'true_label' : row['Sarcasm'],
        'original_pred' : orig_pred,
        'perturbed_pred' : pert_pred,
        'original_conf' : round(orig_conf, 4),
        'perturbed_conf' : round(pert_conf, 4),
        'conf_change' : round(pert_conf - orig_conf, 4),
        'prediction_flip' : orig_pred != pert_pred
    })

results_df = pd.DataFrame(results)

In [42]:
print("---Adversarial Attack Results---")
print(f"Total sentences tested: {len(results_df)}")
print(f"Prediction flips: {results_df['prediction_flip'].sum()}")
print(f"Flip rate: {round(results_df['prediction_flip'].mean() * 100, 2)}%")
print(f"\nAverage confidence change: {round(results_df['conf_change'].mean(), 4)}")
print(f"Average confidence change en-AU: {round(results_df[results_df['variety'] == 'en-AU']['conf_change'].mean(), 4)}")
print(f"Average confidence change en-IN: {round(results_df[results_df['variety'] == 'en-IN']['conf_change'].mean(), 4)}")

---Adversarial Attack Results---
Total sentences tested: 121
Prediction flips: 10
Flip rate: 8.26%

Average confidence change: 0.0018
Average confidence change en-AU: -0.0133
Average confidence change en-IN: 0.0335


In [43]:
flips = results_df[results_df['prediction_flip'] == True].reset_index(drop=True)
all_slang_reset = all_slang.reset_index(drop=True)

print("--- Examples of Prediction Flips ---")
for i, row in flips.iterrows():
    original = all_slang_reset.iloc[i]['text']
    perturbed = replace_slang(original)
    print(f"Example {i+1} [{row['variety']}]")
    print(f"Original: {original[:100]}")
    print(f"Perturbed: {perturbed[:100]}")
    print(f"True label: {'Sarcastic' if row['true_label'] == 1 else 'Not Sarcastic'}")
    print(f"Original pred: {'Sarcastic' if row['original_pred'] == 1 else 'Not Sarcastic'} ({row['original_conf']})")
    print(f"Perturbed pred: {'Sarcastic' if row['perturbed_pred'] == 1 else 'Not Sarcastic'} ({row['perturbed_conf']})")
    print()

--- Examples of Prediction Flips ---
Example 1 [en-AU]
Original: Absolutely lovely service that was down to earth and served with a smile. Subs were well made with d
Perturbed: absolutely lovely service that was down to earth and served with a smile. subs were well made with d
True label: Sarcastic
Original pred: Sarcastic (0.9939)
Perturbed pred: Not Sarcastic (0.8634)

Example 2 [en-AU]
Original: There's heaps of stuff to see just in Victoria  start with some weekend trips. 
Back when I was a yo
Perturbed: there's very of stuff to see just in victoria  start with some weekend trips. 
back when i was a you
True label: Not Sarcastic
Original pred: Sarcastic (0.8488)
Perturbed pred: Not Sarcastic (0.8562)

Example 3 [en-AU]
Original: Should have ditched trams on all but a few routes decades ago and replaced with better buses and bus
Perturbed: should have ditched trams on all but a few routes decades ago and replaced with better buses and bus
True label: Not Sarcastic
Original pred: Sar